# Agent CTF : essai sur GPU gratuit (Kaggle ou Colab)

Ce notebook installe Ollama, télécharge un modèle ouvert, récupère l'agent depuis GitHub et le lance
sur les challenges de démonstration. Il ne coûte rien et ne demande aucune clé d'API.

**Avant de lancer**
- Kaggle : *Settings* → *Accelerator* = **GPU T4 x2**, et *Internet* = **On** (compte vérifié par téléphone).
- Colab : *Exécution* → *Modifier le type d'exécution* → **GPU T4**.

Puis *Run all* (Kaggle) ou *Exécution* → *Tout exécuter* (Colab). Seule la cellule de paramètres se modifie.

In [ ]:
# --- Paramètres (seule cellule à modifier) -----------------------------------
MODEL = "qwen2.5:7b"          # essai suivant : "qwen2.5:14b" (tient aussi sur une T4 de 16 Go)
BRANCH = "main"               # branche du dépôt à tester
CHALLENGES_DIR = "tests/fixtures"  # ou un dossier de challenges, p. ex. "/kaggle/input/mes-challenges"
MAX_STEPS = 20                # budget d'actions du LLM par challenge
REPO = "https://github.com/Chris-Lorys/LLM-CTF-Attack.git"

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Installer et démarrer Ollama

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, urllib.request

env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192", OLLAMA_KEEP_ALIVE="1h")
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama est prêt.")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read()[-2000:])
    raise RuntimeError("Ollama n'a pas démarré : voir le journal ci-dessus")

In [ ]:
!ollama pull {MODEL}

## 3. Récupérer l'agent et lancer les tests

In [ ]:
import os
os.chdir("/content" if os.path.isdir("/content") else "/kaggle/working")
!rm -rf LLM-CTF-Attack && git clone -q -b {BRANCH} {REPO}
os.chdir("LLM-CTF-Attack")
!git log --oneline -1
!pip -q install pytest
!python -m pytest -q

## 4. Lancer l'agent sur les challenges

In [ ]:
!python -m csaw_agent run-all {CHALLENGES_DIR} --model ollama:{MODEL} --max-steps {MAX_STEPS} --runs-dir runs

## 5. Vérifier les traces et afficher un rapport

In [ ]:
import glob, json
from IPython.display import Markdown, display

for trace in sorted(glob.glob("runs/*/*/trace.jsonl")):
    print(trace)
    !python -m csaw_agent verify-trace {trace}

bilan = json.load(open("runs/bilan.json"))
for r in bilan:
    print(f"{r['status']:<20} {r.get('llm_steps', 0):>3} action(s)  {r['challenge']}")

premier = next((r for r in bilan if r.get("llm_steps")), bilan[0])
display(Markdown(open(premier["report"]).read()))

## 6. Récupérer les résultats

L'archive contient, pour chaque challenge, la trace non retouchée (`trace.jsonl`), le rapport
(`rapport.md`) et le bilan global. Sur Kaggle, elle apparaît dans l'onglet *Output* ; sur Colab,
le téléchargement démarre tout seul.

In [ ]:
import shutil
archive = shutil.make_archive("../runs-" + MODEL.replace(":", "-"), "zip", "runs")
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass